# Hyperparameter Optimization

In [1]:
import seaborn as sns
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
import optuna.visualization as vis

from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

c:\Users\acbriza\anaconda3\envs\dpncf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load / Reload Selection Utility Functions

In [136]:
from utils2.optimization import *

----

## Read Config File

In [128]:
config_path = Path(r'experiments\binary')
config_file = config_path / "optimization_config_dev.yml"
#config_file = config_path / "optimization_config_final.yml"
config_dict = ymlconfig.load_config(config_file)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - hyperparameter optimization(final experiment)',
  'classification_type': 'binary',
  'stage': 'hyperparameter_optimization',
  'tag': 'development',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'name': 'catboost'},
 'param_space': {'iterations': {'min': 100, 'max': 500},
  'depth': {'min': 4, 'max': 10},
  'learning_rate': {'min': 0.01, 'max': 0.1},
  'l2_leaf_reg': {'min': 1, 'max': 9}},
 'optimization': {'scoring': 'roc_auc',
  'k_splits_outer': 3,
  'n_repeats_outer': 2,
  'k_splits_inner': 3,
  'n_iter': 5},
 'evaluation': {'confidence': 0.95},
 'final_training': {'k_splits_inner': 3, 'n_iter': 5}}

## Data Loading

In [4]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

## CatBoost BayesSearchCV Optimization  

----

In [11]:
from utils2.optimization import *

### Param Space Definition

In [ ]:
# this one works, but actually performs like grid search
param_space = {
    'iterations': [10], #[100, 200, 500],
    'depth': [4, 6], # [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05], #[0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3], # [1, 3, 5, 7, 9]
}

# correct implementation but not tried
from skopt.space import Integer, Real
param_space = {
    'iterations': Integer(100, 500),
    'depth': Integer(4, 10),
    'learning_rate': Real(0.01, 0.1, prior='log-uniform'),
    'l2_leaf_reg': Integer(1, 9),
}


### Optimization 

In [ ]:
model = CatBoostClassifier(
    verbose=0,  # Show progress every 100 iterations
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=42,
    thread_count=-1  # Use all CPU cores
)

opt_results = nested_cv_youden(
    X=X.values,
    y=y.values,
    model=model,
    param_space=param_space,
    n_splits_outer=5,
    n_repeats_outer=5,
    n_splits_inner=3,
    n_iter=30
)

opt_results

### Calculate Confidence Interval 

In [ ]:
import numpy as np

def mean_confidence_interval(opt_results, metric, confidence=0.95):
    scores = [fold[metric] for fold in opt_results['folds']]
    
    scores = np.array(scores)
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    stderr = std / np.sqrt(n)

    z = 1.96  # 95% normal approx
    margin = z * stderr

    return {
        "mean": mean,
        "std": std,
        "ci_lower": mean - margin,
        "ci_upper": mean + margin,
        "n_folds": n
    }

In [ ]:
youden_ci = mean_confidence_interval(opt_results, "youden")
roc_ci = mean_confidence_interval(opt_results, "roc_auc")

print("Youden 95% CI:", youden_ci)
print("ROC-AUC 95% CI:", roc_ci)

### Train Final Model

#### Define Function for Retraining

In [ ]:
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
import numpy as np
from sklearn.metrics import roc_curve

def train_final_model(X, y, model, param_space, n_splits_inner=3, n_iter=30, random_state=42, n_jobs=-1):
    """
    Train the final deployable model on ALL data:
    1. BayesSearchCV to find best hyperparameters (inner CV on full dataset)
    2. Refit on full dataset with best params
    3. Threshold via Youden index on OOF predictions
    """

    inner_cv = StratifiedKFold(n_splits=n_splits_inner, shuffle=True, random_state=random_state)

    # Step 1: Hyperparameter search on full data
    opt = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        scoring="roc_auc",
        cv=inner_cv,
        n_iter=n_iter,
        n_jobs=n_jobs,
        random_state=random_state,
        refit=True,  # fits final model on all data with best params
    )
    opt.fit(X, y)
    final_model = opt.best_estimator_

    # Step 2: Youden threshold via OOF probabilities on full dataset
    oof_proba = np.zeros(len(y))
    for inner_train_idx, inner_val_idx in inner_cv.split(X, y):
        fold_model = opt.best_estimator_.__class__(**opt.best_params_)
        fold_model.fit(X[inner_train_idx], y[inner_train_idx])
        oof_proba[inner_val_idx] = fold_model.predict_proba(X[inner_val_idx])[:, 1]

    fpr, tpr, thresholds = roc_curve(y, oof_proba)
    best_threshold = float(thresholds[np.argmax(tpr - fpr)])

    return final_model, best_threshold, opt.best_params_


# --- Run nested CV first to get honest performance estimates ---
opt_results = nested_cv_youden(
    X=X.values,
    y=y.values,
    model=model,
    param_space=param_space,
    n_splits_outer=5,
    n_repeats_outer=5,
    n_splits_inner=3,
    n_iter=30
)

print(f"Estimated AUC: {opt_results['roc_auc_mean']:.3f} ± {opt_results['roc_auc_std']:.3f}")


#### Train final model on ALL data

In [ ]:
# --- Then train final model on ALL data ---
final_model, final_threshold, best_params = train_final_model(
    X=X.values, y=y.values, model=model, param_space=param_space
)

print(final_model, final_threshold) 
best_params

### Final Model Prediction

In [ ]:
def predict(X_new, model, threshold):
    proba = model.predict_proba(X_new)[:, 1]
    return (proba >= threshold).astype(int), proba

##### Sample Prediction

In [ ]:
test_size = 20
Xnew = X.iloc[:test_size].values
ypred, ypredproba = predict(Xnew, final_model, final_threshold)
ynew=y.iloc[:test_size].values

cm = confusion_matrix(ynew, ypred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
youden_test = (
    sensitivity + specificity - 1
    if not (np.isnan(sensitivity) or np.isnan(specificity))
    else np.nan
)
roc_auc = (
    roc_auc_score(ypred, ypredproba)
    if len(np.unique(ynew)) > 1
    else np.nan
)

print(youden_test, roc_auc)

---

## Optuna Bayes Search Optimization

In [168]:
config.param_space

namespace(iterations=namespace(min=100, max=500),
          depth=namespace(min=4, max=10),
          learning_rate=namespace(min=0.01, max=0.1),
          l2_leaf_reg=namespace(min=1, max=9))

In [169]:
model_class[config.model.name]

catboost.core.CatBoostClassifier

In [170]:
config.optimization

namespace(scoring='roc_auc',
          k_splits_outer=3,
          n_repeats_outer=2,
          k_splits_inner=3,
          n_iter=5)

In [171]:
def param_space_fn(trial):
    return  {
        "iterations": trial.suggest_int(
            "iterations", 
            config.param_space.iterations.min, 
            config.param_space.iterations.max),
        "depth": trial.suggest_int(
            "depth", 
            config.param_space.depth.min, 
            config.param_space.depth.max),
        "learning_rate": trial.suggest_float(
            "learning_rate", 
            config.param_space.learning_rate.min, 
            config.param_space.learning_rate.max, 
            log=True),
        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", 
            config.param_space.l2_leaf_reg.min, 
            config.param_space.l2_leaf_reg.max),
    }


In [172]:
opt_results = nested_cv_youden_optuna(
    X=X.values,
    y=y.values,
    model_class=model_class[config.model.name],   # class, not an instance
    param_space_fn=param_space_fn,
    n_splits_outer=config.optimization.k_splits_outer,
    n_repeats_outer=config.optimization.n_repeats_outer,
    n_splits_inner=config.optimization.k_splits_inner,
    n_iter=config.optimization.n_iter,   # Optuna trials per outer fold
    random_state=config.experiment.random_seed,
)

In [173]:
opt_results

{'roc_auc_mean': 0.9692212825933756,
 'roc_auc_std': 0.018776054590824542,
 'youden_mean': 0.7956483439041578,
 'youden_std': 0.08021401914153416,
 'sensitivity_mean': 0.9039816772374912,
 'specificity_mean': 0.8916666666666666,
 'threshold_mean': 0.6488591221384802,
 'threshold_std': 0.14354802298703395,
 'folds': [{'fold': 0,
   'roc_auc': 0.9840909090909091,
   'youden': 0.9090909090909092,
   'sensitivity': 0.9090909090909091,
   'specificity': 1.0,
   'threshold': 0.7137815889585889,
   'best_params': {'iterations': 433,
    'depth': 5,
    'learning_rate': 0.015199348301309814,
    'l2_leaf_reg': 2}},
  {'fold': 1,
   'roc_auc': 0.9755813953488373,
   'youden': 0.6499999999999999,
   'sensitivity': 1.0,
   'specificity': 0.65,
   'threshold': 0.3794176963815851,
   'best_params': {'iterations': 231,
    'depth': 10,
    'learning_rate': 0.04635431984752397,
    'l2_leaf_reg': 5}},
  {'fold': 2,
   'roc_auc': 0.9348837209302325,
   'youden': 0.8069767441860465,
   'sensitivity': 0

### Calculate Confidence Interval 

In [174]:
opt_ci  = mean_confidence_interval(opt_results, config)
opt_ci

youden 95.0% CI: {'mean': 0.7956483439041578, 'std': 0.08787005542393887, 'ci_lower': 0.7253389480563618, 'ci_upper': 0.8659577397519539, 'n_folds': 6}
roc_auc 95.0% CI: {'mean': 0.9692212825933756, 'std': 0.020568137280686065, 'ci_lower': 0.9527636475214215, 'ci_upper': 0.9856789176653297, 'n_folds': 6}


{'youden': {'mean': 0.7956483439041578,
  'std': 0.08787005542393887,
  'ci_lower': 0.7253389480563618,
  'ci_upper': 0.8659577397519539,
  'n_folds': 6},
 'roc_auc': {'mean': 0.9692212825933756,
  'std': 0.020568137280686065,
  'ci_lower': 0.9527636475214215,
  'ci_upper': 0.9856789176653297,
  'n_folds': 6}}

### Optimization Summary

In [175]:
opt_results_summary = {
    'youden': f'{opt_results["youden_mean"]:.3f} +/ {opt_results["youden_std"]:.3f}',
    'youden ci': f'{opt_ci["youden"]["ci_lower"]:.3f} - {opt_ci["youden"]["ci_upper"]:.3f}', 
    'roc_auc': f'{opt_results["roc_auc_mean"]:.3f} +/ {opt_results["roc_auc_std"]:.3f}',
    'roc_auc ci': f'{opt_ci["roc_auc"]["ci_lower"]:.3f} - {opt_ci["roc_auc"]["ci_upper"]:.3f}', 
    'threshold': f'{opt_results["threshold_mean"]:.3f} +/ {opt_results["threshold_std"]:.3f}',
    'specificity mean': f'{opt_results["specificity_mean"]:.3f}',
    'sensitivity mean': f'{opt_results["sensitivity_mean"]:.3f}',
}

opt_results_df = pd.DataFrame(opt_results_summary, index=['value']).T
opt_results_df

,value
youden,0.796 +/ 0.080
youden ci,0.725 - 0.866
roc_auc,0.969 +/ 0.019
roc_auc ci,0.953 - 0.986
threshold,0.649 +/ 0.144
specificity mean,0.892
sensitivity mean,0.904


 -----

### Train final model on ALL data

In [176]:
catboost_model = CatBoostClassifier(
    verbose=0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=config.experiment.random_seed, 
    thread_count=-1
)

In [177]:
final_model, final_threshold, best_params = train_final_model(
    X=X.values, 
    y=y.values, 
    model=catboost_model, 
    param_space=param_space,
    n_splits_inner=config.final_training.k_splits_inner,
    n_iter=config.final_training.n_iter, 
    random_state=config.experiment.random_seed, 
    n_jobs=1
)

0:	learn: 0.6763606	total: 3.1ms	remaining: 1.21s
1:	learn: 0.6604786	total: 5.43ms	remaining: 1.06s
2:	learn: 0.6440255	total: 7.45ms	remaining: 964ms
3:	learn: 0.6289632	total: 9.49ms	remaining: 918ms
4:	learn: 0.6151635	total: 11.7ms	remaining: 900ms
5:	learn: 0.6008200	total: 17.7ms	remaining: 1.14s
6:	learn: 0.5845280	total: 20.2ms	remaining: 1.11s
7:	learn: 0.5734462	total: 22.7ms	remaining: 1.08s
8:	learn: 0.5599572	total: 25.1ms	remaining: 1.07s
9:	learn: 0.5466851	total: 27.6ms	remaining: 1.05s
10:	learn: 0.5349979	total: 30.7ms	remaining: 1.06s
11:	learn: 0.5255899	total: 33.1ms	remaining: 1.04s
12:	learn: 0.5159086	total: 35.1ms	remaining: 1.02s
13:	learn: 0.5032802	total: 37.2ms	remaining: 1s
14:	learn: 0.4942683	total: 39.2ms	remaining: 982ms
15:	learn: 0.4836303	total: 41.2ms	remaining: 967ms
16:	learn: 0.4756781	total: 43.4ms	remaining: 955ms
17:	learn: 0.4673219	total: 46.3ms	remaining: 960ms
18:	learn: 0.4589161	total: 48.6ms	remaining: 951ms
19:	learn: 0.4486130	total

In [178]:
print(best_params, final_threshold) 

OrderedDict({'depth': 6, 'iterations': 391, 'l2_leaf_reg': 8, 'learning_rate': 0.02069186296126172}) 0.6504364254423409


### Final Model Sample Prediction

In [179]:
def test_final_model(model, threshold):
    test_size = 189
    Xnew = X.iloc[:test_size].values
    ypred, ypredproba = model_predict(Xnew, model, threshold)
    ynew=y.iloc[:test_size].values

    cm = confusion_matrix(ynew, ypred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    youden_test = (
        sensitivity + specificity - 1
        if not (np.isnan(sensitivity) or np.isnan(specificity))
        else np.nan
    )
    roc_auc = (
        roc_auc_score(ypred, ypredproba)
        if len(np.unique(ynew)) > 1
        else np.nan
    )

    print(youden_test, roc_auc)

test_final_model(final_model, final_threshold)

1.0 1.0


 -----

### Save artifacts and metrics

In [180]:
outputdir = config_path /  config.experiment.tag / config.experiment.stage
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\development\hyperparameter_optimization


In [181]:
final_model_dict = {
    'name': config.model.name,
    'model': final_model,
    'best_params': best_params,
    'threshold' : final_threshold
}

In [182]:
joblib.dump(opt_results, outputdir / "optimization_results.joblib");
joblib.dump(final_model_dict, outputdir / "final_model_dict.joblib");
opt_results_df.to_csv(outputdir / "optimization_results_summary.csv")

### Load Save Results and Verify

In [183]:
loaded_optimization_results = joblib.load(outputdir / "optimization_results.joblib")
loaded_optimization_results

{'roc_auc_mean': 0.9692212825933756,
 'roc_auc_std': 0.018776054590824542,
 'youden_mean': 0.7956483439041578,
 'youden_std': 0.08021401914153416,
 'sensitivity_mean': 0.9039816772374912,
 'specificity_mean': 0.8916666666666666,
 'threshold_mean': 0.6488591221384802,
 'threshold_std': 0.14354802298703395,
 'folds': [{'fold': 0,
   'roc_auc': 0.9840909090909091,
   'youden': 0.9090909090909092,
   'sensitivity': 0.9090909090909091,
   'specificity': 1.0,
   'threshold': 0.7137815889585889,
   'best_params': {'iterations': 433,
    'depth': 5,
    'learning_rate': 0.015199348301309814,
    'l2_leaf_reg': 2}},
  {'fold': 1,
   'roc_auc': 0.9755813953488373,
   'youden': 0.6499999999999999,
   'sensitivity': 1.0,
   'specificity': 0.65,
   'threshold': 0.3794176963815851,
   'best_params': {'iterations': 231,
    'depth': 10,
    'learning_rate': 0.04635431984752397,
    'l2_leaf_reg': 5}},
  {'fold': 2,
   'roc_auc': 0.9348837209302325,
   'youden': 0.8069767441860465,
   'sensitivity': 0

In [184]:
loaded_final_model_dict = joblib.load(outputdir / "final_model_dict.joblib")
loaded_final_model_dict

{'name': 'catboost',
 'model': <catboost.core.CatBoostClassifier at 0x1b03f986cc0>,
 'best_params': OrderedDict([('depth', 6),
              ('iterations', 391),
              ('l2_leaf_reg', 8),
              ('learning_rate', 0.02069186296126172)]),
 'threshold': 0.6504364254423409}

In [185]:
test_final_model(
    loaded_final_model_dict["model"], 
    loaded_final_model_dict["threshold"]) 

1.0 1.0


In [186]:
pd.read_csv(outputdir / "optimization_results_summary.csv")

,Unnamed: 0,value
0,youden,0.796 +/ 0.080
1,youden ci,0.725 - 0.866
2,roc_auc,0.969 +/ 0.019
3,roc_auc ci,0.953 - 0.986
4,threshold,0.649 +/ 0.144
5,specificity mean,0.892
6,sensitivity mean,0.904
